# H Diffusivity Workflow — Script Generator + Analysis

## Workflow overview

This notebook is a **script generator**. Running cells 1–3 writes two files to disk:
- `diffusivity_run.py` — standalone Python orchestrator (Phases 1–3)
- `diffusivity_run.sh` — SLURM submission script configured by notebook variables

Submitting `diffusivity_run.sh` runs the full workflow on the cluster unattended.  
Cells 4+ are run **locally after the cluster jobs complete** to inspect and visualise results.

---

**Phase 1a — Bare bulk minimisation** *(multigpu partition)*  
&nbsp;&nbsp; CG-minimise input structure → `bulk_min.lammps`

**Phase 1b — Temperature-dependent lattice equilibration** *(multigpu partition)*  
&nbsp;&nbsp; For each T: NPT MD → equilibrated structure + a₀(T)  
&nbsp;&nbsp; Insert *n* H atoms at octahedral sites using a₀(T)  
&nbsp;&nbsp; Minimise bulk+H → `{T}K/bulk_min_h.lammps` (one per temperature)

**Phase 2 — NVT MD** *(chained GPU jobs, one per temperature)*  
&nbsp;&nbsp; Equilibration + production NVT with H-atom MSD tracking.  
&nbsp;&nbsp; Self-resubmitting if wall-time is exceeded.

**Phase 3 — Diffusivity extraction** *(inside orchestrator, after all NVT jobs)*  
&nbsp;&nbsp; MSD → D(T) via `run_diffusivity_pipeline` → `diffusivity_table.txt`  
&nbsp;&nbsp; Arrhenius fit → Ea, D₀ via `run_arrhenius_pipeline`

**Phase 4 — Local analysis & visualisation** *(run in this notebook)*  
&nbsp;&nbsp; 4a. Load `diffusivity_table.txt` → summary DataFrame  
&nbsp;&nbsp; 4b. Minimisation QC — convergence & Fmax check  
&nbsp;&nbsp; 4c. NVT thermo QC — temperature drift, MSD traces  
&nbsp;&nbsp; 4d. D vs T plot (log scale, per structure)  
&nbsp;&nbsp; 4e. Ea vs n_H plot  
&nbsp;&nbsp; 4f. Arrhenius overlay (log D vs 1000/T, all n_H)

## Cells 1–2: Imports & configuration

Edit `input_structures`, `n_h_values`, `temperatures`, `NVT_WALL_TIME` here before generating scripts.

In [2]:
import os
import sys

# Add parent directory to path
parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

In [3]:
# ── Imports ───────────────────────────────────────────────────────────────────
from models.config import (
    LAMMPS_CMD, MACE_MODEL_LAMMPS, KOKKOS_FLAGS,
    PAIR_STYLE, PAIR_SUFFIX,
    E2T_7, MASSES_7, ELEM_STR_7,
    SLURM_DEFAULTS, BASE_DIR,
)
#from models.structure import get_lattice_parameter , insert_hydrogen
from models.utils import make_run_dirs
from models.lammps_script import (
    write_minimization_script,
    write_nvt_bulk_script,
    write_nvt_bulk_restart_script,
)
from models.create_slurm import (
    write_slurm_job,
    write_chained_slurm_job,
    submit_slurm_job,
    wait_for_jobs,
)
from models.diffusivity_post_processing import (
    run_diffusivity_pipeline,
    save_diffusivity_table,
    run_arrhenius_pipeline,
)

# ── User-editable config ──────────────────────────────────────────────────────
# BASE_DIR = '/projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel'  
WORK_DIR = os.path.join(BASE_DIR, 'calculation') # '/projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation'

input_structures = [
    # Add full paths to pre-built bulk .lammps files:
    # '/projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/structures/seed1.lammps',
    # '/projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/structures/seed2.lammps',
    '/projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation',
]

n_h_values   = [1, 3, 5, 7, 10]
temperatures = [300, 400, 500, 600, 700, 800]  # K

# Wall time per NVT chained leg (multigpu max ~24h; leave headroom for resubmit)
NVT_WALL_TIME = '24:00:00'
CUTOFF        = '23:55:00'   # timeout inside the chained job before resubmit

# SLURM config for all GPU jobs (minimization + NVT)
GPU_PARTITION = 'multigpu'
GPU_TIME      = NVT_WALL_TIME
GPU_SLURM_CFG = dict(SLURM_DEFAULTS, partition=GPU_PARTITION, time=GPU_TIME)

# SLURM config for the orchestrator script
ORCH_PARTITION      = 'west'
ORCH_CPUS_PER_TASK  = 4
ORCH_MEM            = '16G'
ORCH_TIME           = None  # set to a SLURM time string like '02:00:00' to enable
ORCH_JOB_NAME       = 'diffusivity_orch'
ORCH_OUTPUT         = 'diffusivity_orch_%j.out'
ORCH_LD_PATHS       = GPU_SLURM_CFG['ld_paths']
ORCH_OPENMPI_VER    = GPU_SLURM_CFG['openmpi_ver']
ORCH_CUDA_VERSION   = GPU_SLURM_CFG['cuda_version']
ORCH_CONDA_ENV      = GPU_SLURM_CFG['conda_env']

# MD simulation defaults (sourced from models — override below if needed)
from models import config as _config
from models import lammps_script as _lmp

# Pull authoritative defaults from models/config.py and lammps_script.py
TIMESTEP_PS   = 0.0005    # ps (0.5 fs)
TAU_T_PS      = 0.1       # thermostat damping time in ps
N_EQUIL_STEPS = 500000    # number of equilibration steps
N_PROD_STEPS  = 1500000   # number of production steps
THERMO_EVERY  = 1000      # thermo output frequency (steps)
DUMP_EVERY    = 1000      # trajectory dump frequency (steps)
VELOCITY_SEED = 42        # velocity init seed
RESTART_EVERY = 10000     # restart frequency (steps)

# Allow the user to override any of the above by reassigning the variables

KK = ' '.join(KOKKOS_FLAGS)   # kokkos flags as a single string for shell commands

print('Config loaded.')
print(f'  WORK_DIR         : {WORK_DIR}')
print(f'  input_structures : {len(input_structures)} file(s)')
print(f'  n_h_values       : {n_h_values}')
print(f'  temperatures     : {temperatures}')
print(f'  NVT_WALL_TIME    : {NVT_WALL_TIME}  |  CUTOFF: {CUTOFF}')
print(f'  GPU_PARTITION    : {GPU_PARTITION}')
print(f'  ORCH_PARTITION   : {ORCH_PARTITION}')
print(f'  ORCH_CPUS_PER_TASK: {ORCH_CPUS_PER_TASK}')
print(f'  ORCH_MEM         : {ORCH_MEM}')
print(f'  ORCH_TIME        : {ORCH_TIME}')
print(f'  TIMESTEP_PS      : {TIMESTEP_PS} ps')
print(f'  TAU_T_PS         : {TAU_T_PS} ps')
print(f'  N_EQUIL_STEPS    : {N_EQUIL_STEPS}')
print(f'  N_PROD_STEPS     : {N_PROD_STEPS}')
print(f'  THERMO_EVERY     : {THERMO_EVERY}')
print(f'  DUMP_EVERY       : {DUMP_EVERY}')
print(f'  VELOCITY_SEED    : {VELOCITY_SEED}')
print(f'  RESTART_EVERY    : {RESTART_EVERY}')


Config loaded.
  WORK_DIR         : /projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation
  input_structures : 1 file(s)
  n_h_values       : [1, 3, 5, 7, 10]
  temperatures     : [300, 400, 500, 600, 700, 800]
  NVT_WALL_TIME    : 24:00:00  |  CUTOFF: 23:55:00
  GPU_PARTITION    : multigpu
  ORCH_PARTITION   : west
  ORCH_CPUS_PER_TASK: 4
  ORCH_MEM         : 16G
  ORCH_TIME        : None
  TIMESTEP_PS      : 0.0005 ps
  TAU_T_PS         : 0.1 ps
  N_EQUIL_STEPS    : 500000
  N_PROD_STEPS     : 1500000
  THERMO_EVERY     : 1000
  DUMP_EVERY       : 1000
  VELOCITY_SEED    : 42
  RESTART_EVERY    : 10000


## Cell 3: Generate `diffusivity_run.py`

Builds the full Phase 1→2→3 orchestrator script as a Python string and writes it to disk.  
The script is parameterised by the config values set in Cell 2 (injected as literals via f-string header).

### Phase 1 — Structure preparation (inside `diffusivity_run.py`)

For each `(struct, n_h)`:

**1a.** Write + submit `minimize_bare.lammps` → wait → minimised bulk (`bulk_min.lammps`)

**1b.** For each temperature T (all submitted in parallel):
- Write + submit `npt_{T}K.lammps` (NPT MD at T K, reads `bulk_min.lammps`) → wait
- `get_lattice_parameter(npt_final_{T}K.lammps)` → a0(T)
- `insert_hydrogen(npt_final_{T}K.lammps, a0=a0(T))` → `{T}K/bulk_h_{n_h}H.lammps`
- Write + submit `minimize_h_{T}K.lammps` → wait → `{T}K/bulk_min_h.lammps`
- Collect `T_to_bulk_h[T]` for Phase 2

In [4]:
from models.diffusivity_workflow import generate_diffusivity_scripts

out_py = generate_diffusivity_scripts(
    input_structures=input_structures,
    n_h_values=n_h_values,
    temperatures=temperatures,
    work_dir=WORK_DIR,
    nvt_wall_time=NVT_WALL_TIME,
    cutoff=CUTOFF,
    gpu_partition=GPU_PARTITION,
    gpu_time=GPU_TIME,
    timestep_ps=TIMESTEP_PS,
    tau_t_ps=TAU_T_PS,
    n_equil_steps=N_EQUIL_STEPS,
    n_prod_steps=N_PROD_STEPS,
    thermo_every=THERMO_EVERY,
    dump_every=DUMP_EVERY,
    velocity_seed=VELOCITY_SEED,
    restart_every=RESTART_EVERY,
    out_py=os.path.join(os.getcwd(), 'diffusivity_run.py'),
)
print(f'Written: {out_py}')


Written: /Users/akinyemi.az/Desktop/PhD_Folder/research/MHI/MD/Molecular_Dynamics/MHI_Nickel/calculation/diffusivity_run.py


### Phase 2 — NVT MD (inside `diffusivity_run.py`)

For each temperature:
- `write_nvt_bulk_script` → fresh-start LAMMPS input
- `write_nvt_bulk_restart_script` → restart-from-checkpoint input
- `write_chained_slurm_job` → self-resubmitting `.sh` (cutoff = `CUTOFF`)
- All temperatures submitted simultaneously; `wait_for_jobs` blocks until all complete

### Phase 3 — Post-processing (inside `diffusivity_run.py`)

After all NVT jobs finish:
- `run_diffusivity_pipeline(dump_file, T)` → MSD → D, D_err, R²
- `save_diffusivity_table` → `analysis/diffusivity_table.txt`
- `run_arrhenius_pipeline` → Ea, D₀, `arrhenius.png`

### (c) NEB — Dissociation Barrier

**NB06b2 — Multi-Pathway NEB Seed 7**  
*Full pipeline: IS pool × FS pool → SLURM NEB array → barrier heatmap*

---

**Setup — Load pools and surface graph**

1. Load intact H₂\* adsorption sites from `H2_site_coords.json` (NB05b)  
   &nbsp;&nbsp; Each entry: `site_id`, `true_label`, centroid coords, $E_\text{ads}(\text{H}_2^*)$, status = `intact`
2. Load H\* adsorption sites from NB06b, filter to $E_\text{ads}(\text{H}^*) < 0$  
   &nbsp;&nbsp; 171 candidate H\* sites → **FS pool**
3. Load surface graph (site–site connectivity from ACAT, NB04b)

---

**Cell A — FS pair enumeration**

From $\binom{171}{2} = 14{,}535$ ordered H\*–H\* pairs, apply three filters:

1. H–H separation: $2.5\,\text{Å} \leq d_\text{HH} \leq 6.0\,\text{Å}$  
   &nbsp;&nbsp; Lower bound: avoid unphysical overlap; upper bound: both H\* atoms must be reachable from a single H₂\* IS
2. Graph distance $\geq 2$ (non-adjacent sites — adjacent pairs tend to over-bind)
3. Both sites have $E_\text{ads}(\text{H}^*) < 0$ (thermodynamically stable final states)

Store `true_label1`, `true_label2`, `graph_dist` in each pair dict.  
**Output:** `fs_pairs` — ~7,600 pairs

---

**Cell A+ — FS pair deduplication (shell-1 fingerprint)**

1. Define `site_signature(site)`:  
   &nbsp;&nbsp; Level-1 = sorted multiset of element symbols of the 3–4 nearest surface atoms  
   &nbsp;&nbsp; *(shell-2 removed — empirically shown to give 1.0× compression under MACE-MP-0b2)*
2. Define `pair_signature(p)` = `(site_signature(site1), site_signature(site2))` — order-independent (frozenset of the two)
3. Group `fs_pairs` by `pair_signature` → keep one representative per group, annotate `n_grouped`

**Output:** `unique_pairs` — ~5,000 unique FS pairs (~1.5× reduction)

---

**Cell A++ — IS site deduplication**

1. Apply same `site_signature()` to each H₂\* IS site
2. Group by fingerprint → keep one representative per group, annotate `n_grouped`

**Output:** `unique_is_sites` — ~155 unique IS sites

---

**Cell B — IS × FS cross-product**

Enumerate every `(unique_is_site, unique_pair)` combination.  
Each combination records: `is_true_label`, `fs_true_label1`, `fs_true_label2`, `graph_dist`, full coordinates.  
**Output:** `all_combinations`

---

**Cell C — Proximity filter**

For each combination compute the Euclidean distance between the IS centroid and the midpoint of the two FS H\* sites.  
Retain only combinations with centroid distance $< 5.0\,\text{Å}$.  
**Output:** `filtered_combinations` — **203,150 combinations**

---

**Cell C+ — True-label + graph-distance deduplication** *(key innovation)*

Scientific rationale (empirically verified from 99 NEB results):  
> Sites sharing the same ACAT `true_label` produce **identical barriers to 4 decimal places** under MACE-MP-0b2.  
> The `true_label` already encodes element types of the 3–4 nearest surface atoms + site geometry.  
> Graph distance between the two FS H\* sites must also match — it governs the topological H\*–H\* separation that controls $E_\text{FS}$ and transition-state geometry.

Dedup key per combination:

| Field | What it captures |
|---|---|
| `is_true_label` | IS chemical environment (what H₂\* sits on) |
| `sorted([fs_true_label1, fs_true_label2])` | FS chemistry, order-independent |
| `graph_dist` | Topological H\*–H\* separation |

1. Build `key = (is_true_label, tuple(sorted([fs_true_label1, fs_true_label2])), graph_dist)` for every filtered combination
2. Group by key; keep one representative per group; annotate `n_dedup_group`
3. **Output:** `deduped_combinations` — **914 unique chemical pathways** (203,150 → 914, **222× compression**)

---

**Cell D1 — Write NEB job scripts**

For each of the 914 deduped combinations:

1. Build LAMMPS data files for IS (H₂\* on slab) and FS (2×H\* on slab)
2. Write ASE NEB input script:  
   &nbsp;&nbsp; Phase 1: IDPP interpolation → MDMin pre-relaxation (5 images)  
   &nbsp;&nbsp; Phase 2: CINEB (climbing-image) with MACE-MP-0b2 force calls
3. Write SLURM job script (GPU partition)

**Cell D2 — Write `job_index.txt`**  
One line per job: `(is_true_label, fs_true_label1, fs_true_label2, graph_dist)`

**Cell D3 — Write SLURM array + auto-submit**  
`run_neb_array.sh` (array 0–913) + `auto_submit_nb06b2.sh`

---

**Cell E — Parse results** *(run locally after SLURM jobs complete)*

For each combination in `filtered_combinations`:

1. Read `neb_barrier.txt` → extract $E_a = E_\text{TS} - E_\text{IS}$ and $\Delta E = E_\text{FS} - E_\text{IS}$
2. Build results DataFrame; rank by $E_a$
3. Cross-reference dedup groups to recover statistics for all 203,150 original combinations

---

**Cell F — Visualisation** *(run locally)*

1. **Barrier heatmap:** mean $E_a$ per `(is_true_label, fs_pair_label)` cell
2. **MEP overlay:** all 914 NEB paths as energy vs. image index, coloured by `is_true_label`; top-10 lowest-barrier paths highlighted

## Cell 4: Generate `diffusivity_run.sh` and submit

Writes the SLURM orchestrator script configured by notebook variables (partition, CPU count, memory, and wall-time).
Set `dry_run=False` when ready to submit to the cluster.


---
## Phase 4: Local analysis (run after cluster jobs complete)

### 4a. Load results — `diffusivity_table.txt` → summary DataFrame

### 4b–4c. QC checks — minimisation convergence + NVT temperature drift & MSD traces

In [5]:
from models.diffusivity_workflow import generate_orchestrator_sh

out_sh = generate_orchestrator_sh(
    orch_job_name=ORCH_JOB_NAME,
    orch_partition=ORCH_PARTITION,
    orch_cpus_per_task=ORCH_CPUS_PER_TASK,
    orch_mem=ORCH_MEM,
    orch_time=ORCH_TIME,
    orch_openmpi_ver=ORCH_OPENMPI_VER,
    orch_cuda_version=ORCH_CUDA_VERSION,
    orch_conda_env=ORCH_CONDA_ENV,
    orch_ld_paths=ORCH_LD_PATHS,
    work_dir=os.getcwd(),
    out_py=out_py,
    out_sh=os.path.join(os.getcwd(), 'diffusivity_run.sh'),
)
print(f'Written: {out_sh}')

# Set dry_run=False when ready to submit to the cluster.
job_id = submit_slurm_job(out_sh, dry_run=True)
print(f'\nOrchestrator job: {job_id}')


Written: /Users/akinyemi.az/Desktop/PhD_Folder/research/MHI/MD/Molecular_Dynamics/MHI_Nickel/calculation/diffusivity_run.sh
[dry-run] sbatch /Users/akinyemi.az/Desktop/PhD_Folder/research/MHI/MD/Molecular_Dynamics/MHI_Nickel/calculation/diffusivity_run.sh

Orchestrator job: None


In [6]:
from models.diffusivity_workflow import load_diffusivity_results

RESULTS_ROOT = os.path.join(WORK_DIR, 'results')
summary = load_diffusivity_results(input_structures, n_h_values, RESULTS_ROOT)


  [MISSING] /projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation/results/calculation_1H/analysis/diffusivity_table.txt
  [MISSING] /projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation/results/calculation_3H/analysis/diffusivity_table.txt
  [MISSING] /projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation/results/calculation_5H/analysis/diffusivity_table.txt
  [MISSING] /projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation/results/calculation_7H/analysis/diffusivity_table.txt
  [MISSING] /projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation/results/calculation_10H/analysis/diffusivity_table.txt
No completed results found. Run the cluster jobs first.


In [7]:
from models.diffusivity_workflow import qc_minimizations

qc_minimizations(input_structures, n_h_values, RESULTS_ROOT)


TypeError: unsupported operand type(s) for |: 'types.GenericAlias' and 'NoneType'

In [ ]:
from models.diffusivity_workflow import qc_nvt_thermo

qc_nvt_thermo(input_structures, n_h_values, temperatures, RESULTS_ROOT, timestep_ps=TIMESTEP_PS)


In [ ]:
from models.diffusivity_workflow import plot_D_vs_T

plot_D_vs_T(summary, RESULTS_ROOT)


In [ ]:
from models.diffusivity_workflow import plot_Ea_vs_nH

arr_df = plot_Ea_vs_nH(input_structures, n_h_values, RESULTS_ROOT)


In [ ]:
from models.diffusivity_workflow import plot_arrhenius_overlay

plot_arrhenius_overlay(summary, arr_df, temperatures, n_h_values, RESULTS_ROOT)
